# f6_m00_preparacion.ipynb

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M00 — Preparación |

---

## 🎯 Qué hace

Construye `meta_test.parquet` — tabla de metadatos del conjunto de test para uso
en todos los módulos de Fase 6. Cruza los índices de test con `df_eda_final` para
recuperar `titulacion`, `rama`, `per_id_ficticio` y otras variables de contexto
que no están en `X_test_prep`. Debe ejecutarse **antes** de cualquier otro módulo de Fase 6.



📌 **Genera también** `metricas_modelo.json` con métricas del modelo + cifras canónicas del test (`n_test=6.596`, filtrado 2010-2020). Este JSON es la **única fuente de verdad** para los datos mostrados en la app Streamlit (Fase 7) — sin valores hardcodeados.

## 📋 Requisitos

- `data/04_eda/df_eda_final.parquet` — dataset EDA con titulación y contexto
- `data/05_modelado/X_test_prep.parquet` — índices del conjunto de test (6.725 filas)
- `data/03_features/df_exp_automl_target.parquet` — con `per_id_ficticio`
- `data/03_features/df_alumno_limpio.parquet` — datos a nivel alumno
- `data/05_modelado/y_test.parquet` — etiquetas reales del test
- Entorno: `tfm_abandono` (pandas, numpy)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/06_evaluacion/meta_test.parquet` | Metadatos del test: titulación, rama, per_id_ficticio, abandono, flags |

## 🔄 Flujo

```
data/04_eda/df_eda_final.parquet          ┐
data/05_modelado/X_test_prep.parquet      ├─→ merge por índice → meta_test.parquet
data/03_features/df_exp_automl_target     ┘
    ↓ Verificación de ficheros necesarios
    ↓ Extracción de metadatos del test
    ↓ Agregación a nivel alumno
    ↓ Flag de cautela (titulaciones con sesgo temporal)
    ↓ Verificación final de shape e índices
    → data/06_evaluacion/meta_test.parquet
```

## ➡️ Siguiente

`f6_m01a_shap_global.ipynb` — cálculo de valores SHAP sobre el conjunto de test


In [1]:
# ============================================================================
# CELDA 1: IMPORTS Y RUTAS
# ============================================================================
# Detección robusta de ROOT subiendo niveles hasta encontrar src/
# Verifica existencia de los 4 ficheros necesarios antes de continuar
# ============================================================================

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# ROOT detection estándar TFM
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config_entorno import RAMAS_NOMBRES, NOMBRES_LEGIBLES_FEATURES

# Rutas de datos
RUTA_EDA   = ROOT / 'data' / '04_eda'
RUTA_FEAT  = ROOT / 'data' / '03_features'
RUTA_MODEL = ROOT / 'data' / '05_modelado'

RUTA_EVAL = ROOT / 'data' / '06_evaluacion'
RUTA_EVAL.mkdir(exist_ok=True)  # crea la carpeta si no existe

# Verificar que todos los ficheros necesarios existen antes de continuar
ficheros_necesarios = {
    'df_eda_final':         RUTA_EDA   / 'df_eda_final.parquet',
    'X_test_prep':          RUTA_MODEL / 'X_test_prep.parquet',
    'df_exp_automl_target': RUTA_FEAT  / 'df_exp_automl_target.parquet',
    'df_alumno_limpio':     RUTA_FEAT  / 'df_alumno_limpio.parquet',
}

print('Verificando ficheros necesarios:')
todos_ok = True
for nombre, ruta in ficheros_necesarios.items():
    ok = ruta.exists()
    print(f'  {"✅" if ok else "❌"}  {nombre}')
    if not ok:
        todos_ok = False

assert todos_ok, 'Faltan ficheros necesarios. Revisar rutas.'
print(f'\n✅ Todos los ficheros encontrados')
print(f'ROOT: {ROOT}')

Verificando ficheros necesarios:
  ✅  df_eda_final
  ✅  X_test_prep
  ✅  df_exp_automl_target
  ✅  df_alumno_limpio

✅ Todos los ficheros encontrados
ROOT: C:\proyectos\AU_UJI


In [2]:
# ============================================================================
# CELDA 2: CARGA DE DATOS
# ============================================================================
# df_eda_final: 33.621 filas, índice 0-33620, contiene titulacion
# df_exp:       33.621 filas, mismo índice posicional — aporta per_id_ficticio
# df_lim:       109.568 filas (nivel curso académico) — aporta curso_aca_ini,
#               curso_aca, vive_fuera, cupo. Se agrega a nivel alumno en Celda 4.
# X_test:       6.725 filas — sus índices son posiciones en df_eda_final
# ============================================================================

# Dataset EDA final — contiene titulacion, alineado con modelado
df_eda = pd.read_parquet(RUTA_EDA / 'df_eda_final.parquet')

# Índices del conjunto de test (posiciones en df_eda_final)
X_test = pd.read_parquet(RUTA_MODEL / 'X_test_prep.parquet')

# per_id_ficticio alineado con df_eda_final (índice posicional 0-33620)
df_exp = pd.read_parquet(RUTA_FEAT / 'df_exp_automl_target.parquet')[['per_id_ficticio']]

# Datos de alumno a nivel curso — se agrega por alumno en Celda 4
df_lim = pd.read_parquet(RUTA_FEAT / 'df_alumno_limpio.parquet')[
    ['per_id_ficticio', 'curso_aca_ini', 'curso_aca', 'vive_fuera', 'cupo']
]

print(f'df_eda_final:          {df_eda.shape}  — índice {df_eda.index.min()}-{df_eda.index.max()}')
print(f'X_test:                {X_test.shape}  — índice {X_test.index.min()}-{X_test.index.max()}')
print(f'df_exp (per_id):       {df_exp.shape}')
print(f'df_alumno_limpio:      {df_lim.shape}')

# Verificación crítica: df_exp y df_eda deben estar alineados
assert len(df_exp) == len(df_eda), \
    f'ERROR: df_exp ({len(df_exp)}) y df_eda ({len(df_eda)}) no coinciden'
print('\n✅ df_exp y df_eda alineados (mismo nº filas)')

df_eda_final:          (33621, 26)  — índice 0-33620
X_test:                (6725, 27)  — índice 16-33619
df_exp (per_id):       (33621, 1)
df_alumno_limpio:      (109568, 5)

✅ df_exp y df_eda alineados (mismo nº filas)


In [3]:
# ============================================================================
# CELDA 3: META BASE — extraer contexto de test desde df_eda_final
# ============================================================================
# X_test.index contiene posiciones (ej: 16823, 4904...) en df_eda_final.
# Con iloc extraemos exactamente esas filas y su contexto.
# Añadimos per_id_ficticio desde df_exp con el mismo mecanismo posicional.
# ============================================================================

# Columnas de contexto disponibles en df_eda_final
COLS_EDA = ['titulacion', 'rama', 'sexo', 'pais_nombre',
            'provincia', 'via_acceso', 'abandono']

# Extraer filas de test por posición
meta_base = df_eda.iloc[X_test.index][COLS_EDA].copy()
meta_base.index = X_test.index

# Añadir per_id_ficticio desde df_exp (mismo índice posicional)
meta_base['per_id_ficticio'] = df_exp.iloc[X_test.index]['per_id_ficticio'].values

print(f'meta_base shape:           {meta_base.shape}')
print(f'Titulaciones únicas:       {meta_base["titulacion"].nunique()}')
print(f'per_id_ficticio únicos:    {meta_base["per_id_ficticio"].nunique()}')
print(f'Tasa abandono en test:     {meta_base["abandono"].mean()*100:.1f}%')
print(f'\nPrimeras 3 filas:')
print(meta_base.head(3))

meta_base shape:           (6725, 8)
Titulaciones únicas:       40
per_id_ficticio únicos:    6603
Tasa abandono en test:     29.2%

Primeras 3 filas:
                             titulacion rama    sexo pais_nombre provincia  \
16823  Grado en Finanzas y Contabilidad   SO   Mujer      España  Castelló   
4904                Grado en Psicología   SA  Hombre      España  Castelló   
11906                 Grado en Medicina   SA   Mujer      España  Castelló   

                           via_acceso  abandono  per_id_ficticio  
16823  Pruebas acceso Bachiller Logse         0          1197964  
4904   Pruebas acceso Bachiller Logse         0           733138  
11906  Pruebas acceso Bachiller Logse         0           976958  


In [4]:
# ============================================================================
# CELDA 4: AGREGAR df_alumno_limpio A NIVEL ALUMNO
# ============================================================================
# df_alumno_limpio tiene 109.568 filas (una por alumno×año×titulación).
# Necesitamos una fila por per_id_ficticio para hacer el merge.
#
# Reglas de agregación:
#   curso_aca_ini → min: año del primer registro/relación con UJI
#                        (puede ser anterior a la carrera: cursos verano, idiomas, etc.)
#   curso_aca     → min: año de primera matrícula en una carrera de grado (variable canónica)
#   vive_fuera    → moda: valor más frecuente durante sus años de estudio
#   cupo          → moda: tipo de cupo de acceso predominante
# ============================================================================

df_extra = (
    df_lim
    .groupby('per_id_ficticio')
    .agg(
        curso_aca_ini=('curso_aca_ini', 'min'),  # primer año de relación con UJI
        curso_aca    =('curso_aca',     'min'),  # primer año de matrícula en grado (canónica)
        vive_fuera   =('vive_fuera',    lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        cupo         =('cupo',          lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    )
    .reset_index()
)

print(f'df_extra shape: {df_extra.shape}  (un alumno por fila)')
print(f'\nEjemplo primeras 3 filas:')
print(df_extra.head(3))
print(f'\ncupo — distribución:')
print(df_extra['cupo'].value_counts().head(8))

df_extra shape: (30872, 5)  (un alumno por fila)

Ejemplo primeras 3 filas:
   per_id_ficticio  curso_aca_ini  curso_aca  vive_fuera     cupo
0              106           2012       2012        True  General
1              134           2009       2010       False     None
2              426           2010       2010       False  General

cupo — distribución:
cupo
General                   26323
Mayor 25 Años               594
Titulados                   287
Mayor 40años                 35
Mayor 45años                 31
Minusvalidos                 20
Deportistas Alto Nivel       18
Mayor 45 Años                18
Name: count, dtype: int64


In [5]:
# ============================================================================
# CELDA 5: MERGE Y CÁLCULO DE n_titulaciones
# ============================================================================
# Merge meta_base + df_extra por per_id_ficticio.
# n_titulaciones: cuántas carreras distintas cursó cada alumno.
# Se calcula sobre df_eda completo (no solo test) para ser representativo.
# ============================================================================

# Merge principal — añade curso_aca_ini, vive_fuera, cupo
meta_test = meta_base.merge(df_extra, on='per_id_ficticio', how='left')
meta_test.index = X_test.index  # restaurar índice tras merge

# Calcular n_titulaciones: nº de titulaciones distintas por alumno en df_eda
# df_exp tiene per_id_ficticio alineado con df_eda (mismo índice posicional)
df_eda_con_id = df_eda[['titulacion']].copy()
df_eda_con_id['per_id_ficticio'] = df_exp['per_id_ficticio'].values

n_tit_map = (
    df_eda_con_id
    .groupby('per_id_ficticio')['titulacion']
    .nunique()
    .rename('n_titulaciones')
    .reset_index()
)

# Añadir n_titulaciones y restaurar índice
meta_test = meta_test.merge(n_tit_map, on='per_id_ficticio', how='left')
meta_test.index = X_test.index

print(f'meta_test shape: {meta_test.shape}')
print(f'Columnas: {meta_test.columns.tolist()}')
print(f'\nNulos por columna:')
print(meta_test.isnull().sum())
print(f'\nn_titulaciones distribución:')
print(meta_test['n_titulaciones'].value_counts().sort_index())

meta_test shape: (6725, 13)
Columnas: ['titulacion', 'rama', 'sexo', 'pais_nombre', 'provincia', 'via_acceso', 'abandono', 'per_id_ficticio', 'curso_aca_ini', 'curso_aca', 'vive_fuera', 'cupo', 'n_titulaciones']

Nulos por columna:
titulacion           0
rama                 0
sexo                 0
pais_nombre          0
provincia            0
via_acceso           0
abandono             0
per_id_ficticio      0
curso_aca_ini        0
curso_aca            0
vive_fuera           0
cupo               709
n_titulaciones       0
dtype: int64

n_titulaciones distribución:
n_titulaciones
1    5672
2     979
3      68
4       3
5       3
Name: count, dtype: int64


In [6]:
# ============================================================================
# CELDA 6: FLAG DE CAUTELA — titulaciones con sesgo temporal
# ============================================================================
# Se incluyen TODAS las titulaciones. Las que tienen sesgo temporal se marcan
# con flag_cautela para que los módulos de Fase 6 puedan informar al lector.
#
# Criterios:
#   n_test < 30                → muestra insuficiente para métricas estables
#   Plan 2020 con 0% abandono  → alumnos sin tiempo suficiente de abandonar
#   Plan 2018 bajo abandono    → truncamiento temporal parcial
#   0% abandono sin plan nuevo → posible artefacto del dataset
# ============================================================================

TITULACIONES_CAUTELA = {
    'Doble Grado en Administración y Dirección de Empresas y Derecho,': 'n_test<30',
    'Grado en Criminologia y Seguridad  (Plan 2020)':                   'plan_reciente_0pct_abandono',
    'Grado en Arquitectura Técnica (Plan 2020)':                        'plan_reciente_0pct_abandono',
    'Grado en Ingeniería Agroalimentaria y del Medio Rural (Plan 2018)':'plan_reciente',
    'Grado en Maestro en Educación Primaria (Plan 2018)':               'plan_reciente_bajo_abandono',
    'Grado en Maestro en Educación Infantil (Plan 2018)':               'plan_reciente_bajo_abandono',
    'Grado en Ingeniería de la Edificación':                            '0pct_abandono_en_test',
}

meta_test['flag_cautela'] = meta_test['titulacion'].map(TITULACIONES_CAUTELA).fillna('ok')

print('Distribución flag_cautela:')
print(meta_test['flag_cautela'].value_counts())
print(f'\nObservaciones con cautela:   {(meta_test["flag_cautela"] != "ok").sum():,}')
print(f'Observaciones sin cautela:   {(meta_test["flag_cautela"] == "ok").sum():,}')

Distribución flag_cautela:
flag_cautela
ok                             6457
plan_reciente_bajo_abandono     186
0pct_abandono_en_test            41
plan_reciente_0pct_abandono      24
plan_reciente                    17
Name: count, dtype: int64

Observaciones con cautela:   268
Observaciones sin cautela:   6,457


In [7]:
# ============================================================================
# CELDA 7: VERIFICACIÓN FINAL
# ============================================================================
# Comprueba shape, índices y columnas esperadas antes de guardar.
# Si y_test.parquet existe, verifica que abandono coincide con el target real.
# ============================================================================

print('=' * 60)
print('VERIFICACIÓN FINAL DE meta_test')
print('=' * 60)

# Shape correcto
assert meta_test.shape[0] == len(X_test), \
    f'ERROR: {meta_test.shape[0]} filas en meta_test vs {len(X_test)} en X_test'
print(f'✅ Shape correcto: {meta_test.shape}')

# Índice alineado con X_test
assert list(meta_test.index) == list(X_test.index), \
    'ERROR: índices no coinciden con X_test'
print(f'✅ Índices alineados con X_test')

# Columnas esperadas
COLS_ESPERADAS = [
    'titulacion', 'rama', 'sexo', 'pais_nombre', 'provincia',
    'via_acceso', 'abandono', 'per_id_ficticio',
    'curso_aca_ini', 'vive_fuera', 'cupo',
    'n_titulaciones', 'flag_cautela'
]
print(f'\nColumnas:')
for col in COLS_ESPERADAS:
    print(f'  {"✅" if col in meta_test.columns else "❌"}  {col}')

# Verificar abandono vs y_test si existe
ruta_y = RUTA_MODEL / 'y_test.parquet'
if ruta_y.exists():
    y_test = pd.read_parquet(ruta_y)
    coincide = (meta_test['abandono'].values == y_test.values.ravel()).all()
    print(f'\n✅ abandono coincide con y_test: {coincide}')
else:
    print(f'\n⚠️  y_test.parquet no encontrado — abandono tomado de df_eda_final')
    print(f'   Distribución: {meta_test["abandono"].value_counts().to_dict()}')

print(f'\n✅ Verificación completada — listo para guardar')

VERIFICACIÓN FINAL DE meta_test
✅ Shape correcto: (6725, 14)
✅ Índices alineados con X_test

Columnas:
  ✅  titulacion
  ✅  rama
  ✅  sexo
  ✅  pais_nombre
  ✅  provincia
  ✅  via_acceso
  ✅  abandono
  ✅  per_id_ficticio
  ✅  curso_aca_ini
  ✅  vive_fuera
  ✅  cupo
  ✅  n_titulaciones
  ✅  flag_cautela

✅ abandono coincide con y_test: True

✅ Verificación completada — listo para guardar


In [8]:
# ============================================================================
# CELDA 8: GUARDAR meta_test.parquet
# ============================================================================
# Guardado en data/05_modelado/ junto con X_test_prep y y_test.
# Para cargar en cualquier módulo de Fase 6:
#   meta = pd.read_parquet(RUTA_MODEL / 'meta_test.parquet')
# ============================================================================


ruta_salida = RUTA_EVAL / 'meta_test.parquet'
meta_test.to_parquet(ruta_salida)

print(f'✅ Guardado: {ruta_salida}')
print(f'   Tamaño: {ruta_salida.stat().st_size / 1024:.1f} KB')

# Resumen ejecutivo
print(f'\n{"+" * 60}')
print(f'  RESUMEN meta_test.parquet')
print(f'{"+" * 60}')
print(f'  Observaciones en test:      {len(meta_test):,}')
print(f'  Alumnos únicos:             {meta_test["per_id_ficticio"].nunique():,}')
print(f'  Titulaciones:               {meta_test["titulacion"].nunique()} (todas incluidas)')
print(f'  Con 2+ titulaciones:        {(meta_test["n_titulaciones"] > 1).sum():,} '
      f'({(meta_test["n_titulaciones"] > 1).mean()*100:.1f}%)')
print(f'  Con flag cautela:           {(meta_test["flag_cautela"] != "ok").sum():,}')
print(f'  Tasa abandono en test:      {meta_test["abandono"].mean()*100:.1f}%')
print(f'  Cursos de ingreso:          '
      f'{sorted(meta_test["curso_aca_ini"].dropna().unique().astype(int).tolist())}')
print(f'  Viven fuera (%):            {meta_test["vive_fuera"].mean()*100:.1f}%')
print(f'{"+" * 60}')
print(f'\n🎯 meta_test.parquet listo para Fase 6.')

✅ Guardado: C:\proyectos\AU_UJI\data\06_evaluacion\meta_test.parquet
   Tamaño: 118.9 KB

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  RESUMEN meta_test.parquet
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Observaciones en test:      6,725
  Alumnos únicos:             6,603
  Titulaciones:               40 (todas incluidas)
  Con 2+ titulaciones:        1,053 (15.7%)
  Con flag cautela:           268
  Tasa abandono en test:      29.2%
  Cursos de ingreso:          [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
  Viven fuera (%):            12.9%
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

🎯 meta_test.parquet listo para Fase 6.


In [9]:
# ============================================================================
# CELDA 8b: GUARDAR X_test_prep_ids.parquet
# ============================================================================
# Fichero puente: indice posicional de Fase 5 + per_id_ficticio
# Permite join robusto en loaders.py usando per_id en lugar del indice.
# Estable aunque se re-ejecute Fase 5 con nuevos datos Excel.
# NO entra al modelo — solo sirve como puente de union.
# ============================================================================

x_test_ids = meta_test[["per_id_ficticio"]].copy()
x_test_ids.index = X_test.index

ruta_ids = RUTA_MODEL / "X_test_prep_ids.parquet"
x_test_ids.to_parquet(ruta_ids)

print("Guardado:", ruta_ids)
print("Shape:", x_test_ids.shape)
print("per_id_ficticio unicos:", x_test_ids["per_id_ficticio"].nunique())
print("Indice:", x_test_ids.index.min(), "-", x_test_ids.index.max())
print(x_test_ids.head())
print("X_test_prep_ids.parquet listo")


Guardado: C:\proyectos\AU_UJI\data\05_modelado\X_test_prep_ids.parquet
Shape: (6725, 1)
per_id_ficticio unicos: 6603
Indice: 16 - 33619
       per_id_ficticio
16823          1197964
4904            733138
11906           976958
25232          1506036
21637          1426114
X_test_prep_ids.parquet listo


In [10]:
# ============================================================================
# CELDA 10: GUARDAR metricas_modelo.json — para la app Streamlit (Fase 7)
# ============================================================================
# SISTEMA DINÁMICO OPCIÓN C
# -------------------------
# Selecciona el MODELO GANADOR de Fase 5 leyendo `resultados_maestro.parquet`
# y aplicando los criterios definidos en `src/config_modelado.py`:
#
#   1. F1 binario (clase abandono) — métrica principal
#   2. Recall — primer desempate (asimetría coste FN > FP en abandono)
#   3. AUC — segundo desempate (capacidad discriminativa global)
#   4. Tiempo — tercer desempate (eficiencia para producción)
#
# Calcula métricas reales sobre X_test_prep, lee el baseline AutoML del JSON
# real (cargar_baseline_automl) y guarda todo en metricas_modelo.json para
# que la app Streamlit lea sin valores hardcodeados.
#
# ZERO HARDCODES: si re-ejecutas Fase 5 con otro dataset → el JSON se
# regenera automáticamente con el nuevo ganador y nuevas métricas.
# ============================================================================

import json as _json
import joblib
from datetime import datetime
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score, accuracy_score
)

from src.config_modelado import (
    CRITERIO_GANADOR,
    CRITERIO_DESEMPATE_1,
    CRITERIO_DESEMPATE_2,
    CRITERIO_DESEMPATE_3,
    UMBRAL_EMPATE,
    cargar_baseline_automl,
    descripcion_modelo,
)

# ============================================================================
# PASO 1 — Seleccionar el modelo GANADOR del parquet de Fase 5
# ============================================================================

ruta_resultados = RUTA_MODEL / 'results' / 'resultados_maestro.parquet'
assert ruta_resultados.exists(), (
    f'❌ No encontrado: {ruta_resultados}. '
    f'Ejecuta primero Fase 5 (notebooks f5_m01a a f5_m07).'
)

df_resultados = pd.read_parquet(ruta_resultados)
print(f'📊 Resultados Fase 5: {len(df_resultados)} modelos × estrategias')


def seleccionar_ganador(df: pd.DataFrame) -> pd.Series:
    """
    Selecciona el modelo ganador aplicando criterios en cascada.

    Criterios (definidos en src/config_modelado.py):
        1. Mejor F1_test
        2. Empate (diff < UMBRAL_EMPATE) → mejor Recall_test
        3. Empate → mejor AUC_test
        4. Empate → menor tiempo_s (más eficiente)
    """
    candidatos = df.copy()

    # 1) Mejor F1
    f1_max = candidatos[CRITERIO_GANADOR].max()
    candidatos = candidatos[candidatos[CRITERIO_GANADOR] >= f1_max - UMBRAL_EMPATE]

    if len(candidatos) > 1:
        # 2) Desempate por Recall
        r_max = candidatos[CRITERIO_DESEMPATE_1].max()
        candidatos = candidatos[
            candidatos[CRITERIO_DESEMPATE_1] >= r_max - UMBRAL_EMPATE
        ]

    if len(candidatos) > 1:
        # 3) Desempate por AUC
        a_max = candidatos[CRITERIO_DESEMPATE_2].max()
        candidatos = candidatos[
            candidatos[CRITERIO_DESEMPATE_2] >= a_max - UMBRAL_EMPATE
        ]

    if len(candidatos) > 1:
        # 4) Desempate por tiempo (menor = mejor)
        candidatos = candidatos.nsmallest(1, CRITERIO_DESEMPATE_3)

    return candidatos.iloc[0]


ganador = seleccionar_ganador(df_resultados)
modelo_nombre   = ganador['modelo']
estrategia      = ganador['estrategia']
modelo_familia  = ganador['familia']
modelo_pkl_name = f'{modelo_nombre}__{estrategia}.pkl'

print(f'🏆 Modelo ganador: {modelo_nombre} (estrategia: {estrategia})')
print(f'   Familia: {modelo_familia}')
print(f'   F1_test={ganador["f1_test"]:.4f} | AUC_test={ganador["auc_test"]:.4f} | '
      f'Recall_test={ganador["recall_test"]:.4f} | Tiempo={ganador["tiempo_s"]:.1f}s')

# ============================================================================
# PASO 2 — Cargar el .pkl del ganador y datos de test
# ============================================================================

ruta_modelo = RUTA_MODEL / 'models' / modelo_pkl_name
ruta_X_prep = RUTA_MODEL / 'X_test_prep.parquet'
ruta_y      = RUTA_MODEL / 'y_test.parquet'

assert ruta_modelo.exists(), (
    f'❌ El modelo ganador no tiene .pkl en disco: {ruta_modelo}\n'
    f'   Solución: re-ejecuta el notebook de Fase 5 que entrena {modelo_nombre} '
    f'con la estrategia "{estrategia}" para que se guarde el .pkl.'
)
assert ruta_X_prep.exists(), f'❌ No encontrado: {ruta_X_prep}'
assert ruta_y.exists(),      f'❌ No encontrado: {ruta_y}'

modelo           = joblib.load(ruta_modelo)
X_test_prep_eval = pd.read_parquet(ruta_X_prep)
y_test_eval      = pd.read_parquet(ruta_y).values.ravel()

# ============================================================================
# PASO 3 — Predicciones y métricas REALES sobre el split de test
# ============================================================================

prob_test = modelo.predict_proba(X_test_prep_eval)[:, 1]
pred_test = (prob_test >= 0.5).astype(int)

# ============================================================================
# PASO 4 — Estadísticas del dataset (DINÁMICAS, calculadas)
# ============================================================================

n_registros_total = len(df_eda)
n_alumnos_total   = int(df_exp['per_id_ficticio'].nunique())
tasa_abandono_total = (
    float(df_eda['abandono'].mean())
    if 'abandono' in df_eda.columns
    else float(y_test_eval.mean())
)

# Período: calculado de meta_test['curso_aca'] (NO hardcoded)
# Usamos curso_aca (año de primera matrícula en grado) y NO curso_aca_ini
# (que puede incluir relaciones previas con UJI por cursos de verano, idiomas,
# deporte, etc. — NO carrera de grado). Variable canónica para abandono académico.
periodo_ini = int(meta_test['curso_aca'].min())
periodo_fin = int(meta_test['curso_aca'].max())

# Tamaño del test (filtrado por curso_aca, variable canónica del proyecto)
n_test_total    = len(X_test_prep_eval)
n_test_canonico = int(meta_test['curso_aca'].between(periodo_ini, periodo_fin).sum())

# Conteo de features (DINÁMICO, leído de X_test_prep)
n_features_tecnicas = X_test_prep_eval.shape[1]
n_features_missing  = sum(1 for c in X_test_prep_eval.columns if c.endswith('_missing'))
n_features          = n_features_tecnicas - n_features_missing

# ============================================================================
# PASO 5 — Baseline AutoML (DINÁMICO, leído del JSON real)
# ============================================================================

baseline = cargar_baseline_automl()  # Lee data/automl/automl_comparativa_final.json
if baseline is None:
    print('⚠️ Baseline AutoML no disponible — el JSON automl_comparativa_final.json no se encontró')

# ============================================================================
# PASO 6 — Construir el diccionario de métricas (todo dinámico)
# ============================================================================

metricas = {
    # --- Modelo (DINÁMICO, del parquet) ---
    'modelo_nombre':       modelo_nombre,
    'modelo_estrategia':   estrategia,
    'modelo_familia':      modelo_familia,
    'modelo_pkl':          modelo_pkl_name,
    'modelo_descripcion':  descripcion_modelo(modelo_nombre),

    # --- Métricas (DINÁMICAS, calculadas sobre test) ---
    'auc':       round(float(roc_auc_score(y_test_eval, prob_test)), 4),
    'f1':        round(float(f1_score(y_test_eval, pred_test)), 4),
    'precision': round(float(precision_score(y_test_eval, pred_test)), 4),
    'recall':    round(float(recall_score(y_test_eval, pred_test)), 4),
    'accuracy':  round(float(accuracy_score(y_test_eval, pred_test)), 4),

    # --- Criterios de selección (decisiones metodológicas) ---
    'criterio_seleccion':  CRITERIO_GANADOR,
    'criterio_desempate':  CRITERIO_DESEMPATE_1,

    # --- Dataset COMPLETO (DINÁMICO) ---
    'n_registros':       n_registros_total,
    'n_alumnos_unicos':  n_alumnos_total,
    'n_titulaciones':    int(meta_test['titulacion'].nunique()),
    'tasa_abandono':     round(tasa_abandono_total, 4),

    # --- Período (DINÁMICO, calculado de df_eda) ---
    'periodo_inicio':    periodo_ini,
    'periodo_fin':       periodo_fin,

    # --- Test (DINÁMICO) ---
    'n_test_total':      n_test_total,
    'n_test':            n_test_canonico,

    # --- Features (DINÁMICO, leído de X_test_prep) ---
    'n_features':           n_features,
    'n_features_tecnicas':  n_features_tecnicas,
    'n_features_missing':   n_features_missing,

    # --- Baseline AutoML (DINÁMICO, leído del JSON real) ---
    'baseline_nombre':     baseline['modelo']    if baseline else None,
    'baseline_framework':  baseline['framework'] if baseline else None,
    'baseline_dataset':    baseline['dataset']   if baseline else None,
    'baseline_auc':        baseline['auc_roc']   if baseline else None,
    'baseline_f1':         baseline['f1']        if baseline else None,

    # --- Auditoría ---
    'fecha_calculo':       datetime.now().isoformat(timespec='seconds'),
}

# ============================================================================
# PASO 7 — Guardar el JSON
# ============================================================================

ruta_metricas = RUTA_EVAL / 'metricas_modelo.json'
with open(ruta_metricas, 'w', encoding='utf-8') as f:
    _json.dump(metricas, f, ensure_ascii=False, indent=2)

print(f'\n✅ Guardado: {ruta_metricas}\n')
for k, v in metricas.items():
    print(f'  {k}: {v}')


📊 Resultados Fase 5: 69 modelos × estrategias
🏆 Modelo ganador: LightGBM (estrategia: none)
   Familia: Gradient Boosting
   F1_test=0.8334 | AUC_test=0.9564 | Recall_test=0.8048 | Tiempo=25.1s

✅ Guardado: C:\proyectos\AU_UJI\data\06_evaluacion\metricas_modelo.json

  modelo_nombre: LightGBM
  modelo_estrategia: none
  modelo_familia: Gradient Boosting
  modelo_pkl: LightGBM__none.pkl
  modelo_descripcion: Gradient boosting basado en histogramas — rápido y eficiente.
  auc: 0.9564
  f1: 0.8334
  precision: 0.8641
  recall: 0.8048
  accuracy: 0.9059
  criterio_seleccion: f1_test
  criterio_desempate: recall_test
  n_registros: 33621
  n_alumnos_unicos: 30872
  n_titulaciones: 40
  tasa_abandono: 0.2925
  periodo_inicio: 2010
  periodo_fin: 2020
  n_test_total: 6725
  n_test: 6725
  n_features: 24
  n_features_tecnicas: 27
  n_features_missing: 3
  baseline_nombre: CatBoost_BAG_L2
  baseline_framework: AutoGluon
  baseline_dataset: D_strict
  baseline_auc: 0.9365
  baseline_f1: 0.797
  

In [11]:
# ============================================================================
# CELDA 11: REGENERAR HTML INDEX CON LOS DATOS NUEVOS
# ============================================================================
# Tras generar metricas_modelo.json (celda 10), regeneramos automáticamente
# las páginas HTML que dependen de él:
#   - docs/html/index.html         (página principal del proyecto)
#   - docs/html/fase7/fase7_index.html (página de la app Streamlit)
#
# Así, ejecutar "Run All" deja TODO actualizado: JSON + HTMLs.
# Cero pasos manuales. Coherente con el sistema dinámico del proyecto.
# ============================================================================

from src.index_generator import actualizar_index
actualizar_index()

🔄 Actualizando index.html...
  ✅ KPIs: Expedientes: 30.872 | Período: 2010-2020 | Variables: 24 | Abandono: 29,2% | AUC: 0,9564
  📊 Fases: Fase 5: ✅ | Fase 6: ✅ | Fase 7: ✅
  ✅ C:\proyectos\AU_UJI\docs\html\index.html
  ✅ C:\proyectos\AU_UJI\docs\html\fase7\fase7_index.html


WindowsPath('C:/proyectos/AU_UJI/docs/html/index.html')